# `update_tomography()`: full replay vs incremental

With an `ExpectationValue` backend, `update_tomography(incremental=False)` resets the model and replays the whole circuit; the default (`incremental=True`) applies only the gates added since the previous call. Full replay therefore costs O(updates × gates); incremental O(gates).

Workload: a 4×4 grid, k=3, three sweeps of ZZ relationships on every edge, refreshing after each gate (what `set_relationship(update=True)`, the default, does).

In [1]:
import random
import numpy as np
from qiskit.quantum_info import Statevector
from quantumgraph import QuantumGraph, ExpectationValue

ROWS, COLS, K, STEPS = 4, 4, 3, 3
N = ROWS * COLS
EDGES = [(q, q + 1) for q in range(N) if q % COLS != COLS - 1] + [(q, q + COLS) for q in range(N - COLS)]

def prep(incremental, seed=0):
    random.seed(seed)
    g = QuantumGraph(N, coupling_map=EDGES, backend=ExpectationValue(N, k=K, coupling_map=EDGES))
    for q in range(N):
        g.set_bloch({"X": 1}, q, update=False)
    g.update_tomography(incremental=incremental)
    for _ in range(STEPS):
        for a, b in EDGES:
            g.set_relationship({"ZZ": -1}, a, b, fraction=1 / 3, update=False)
            g.update_tomography(incremental=incremental)
    return g

def exact_zz(g):
    """Mean exact <ZZ> over edges from the statevector of g.qc (target is -1)."""
    probs = Statevector(g.qc).probabilities(); idx = np.arange(len(probs))
    z = lambda q: 1 - 2 * ((idx >> q) & 1)
    return float(np.mean([(probs * z(a) * z(b)).sum() for a, b in EDGES]))

In [2]:
%%time
full = prep(incremental=False)

CPU times: user 22.6 s, sys: 401 ms, total: 23 s
Wall time: 22.6 s


In [3]:
%%time
inc = prep(incremental=True)

CPU times: user 887 ms, sys: 180 ms, total: 1.07 s
Wall time: 880 ms


In [4]:
print(f"exact <ZZ>: full {exact_zz(full):.3f}   incremental {exact_zz(inc):.3f}")

exact <ZZ>: full -0.538   incremental -0.553
